# Weekend suite: reviewer-ready comparison
All runs use the categorical semi-synthetic design (real sales target, PoS-like categorical descriptors, planted level step,
effect = snr × null spread σ, outliers, top-20, Jaccard ≥ 0.8). Every method runs on the **same tasks** (paired), and
"wins / losses" count tasks only one of the two methods solved (McNemar exact binomial test).
Reference method: **BS w=10** (beam search, width 10). "BUS" = fixed BUS: scale-free, time-aggregated, non-convex SNN
mini-clusters (expanded), described by enrichment z-scores. Source: `results_suite/<run>/tasks.csv`.

In [1]:
import numpy as np, pandas as pd
from scipy.stats import binomtest
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
KEY = ["k", "lits", "snr", "outliers", "rep"]
RUNS = ["main", "seeds", "depth6", "hier", "walmart", "favorita", "effect", "sensitivity", "beam100"]
data = {}
for r in RUNS:
    d = pd.read_csv(f"results_suite/{r}/tasks.csv"); d["found"] = d["rank"] <= 20; data[r] = d

def table(r, ref="BS w=10", group=None):
    d = data[r] if group is None else data[r].query(group)
    w = d.pivot_table(index=KEY, columns="method", values="found").astype(bool)
    rows = []
    for m in dict.fromkeys(d["method"]):
        g = d[d.method == m]
        row = {"method": m, "found": g.found.mean(), "avg evaluations": g.evaluated.mean(), "avg seconds": g.runtime_s.mean()}
        if m != ref:
            wa, wb = int((w[m] & ~w[ref]).sum()), int((~w[m] & w[ref]).sum())
            row.update({"wins": wa, "losses": wb, "p (McNemar)": binomtest(wa, wa + wb).pvalue if wa + wb else 1.0})
        rows.append(row)
    out = pd.DataFrame(rows).set_index("method")
    out["p (McNemar)"] = out["p (McNemar)"].map(lambda v: "" if pd.isna(v) else f"{v:.1e}")
    return out.round(3)

print({r: data[r].drop_duplicates(KEY).shape[0] for r in RUNS}, "tasks per run")

{'main': 875, 'seeds': 433, 'depth6': 433, 'hier': 474, 'walmart': 440, 'favorita': 455, 'effect': 660, 'sensitivity': 433, 'beam100': 292} tasks per run


## 1. Main comparison (M5 target, 200 / 800 conditions, depth 3–6, 30 repetitions)

In [2]:
display(table("main"))
display(data["main"].pivot_table(index=["k", "lits"], columns="method", values="found").round(2))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=5,0.507,4948.867,4.510,23.0,25.0,8.9e-01
BS w=10,0.510,9203.274,8.716,NaN,NaN,
BS w=20,0.520,17227.071,16.905,14.0,5.0,6.4e-02
BS w=50,0.520,38274.747,34.622,16.0,7.0,9.3e-02
BUS,0.752,173.070,3.639,219.0,7.0,1.0e-55
BUS +local,0.611,3356.817,3.846,93.0,4.0,4.6e-23
pool,0.595,619.234,2.220,81.0,6.0,7.0e-18
itemsets s1,0.752,173.070,1.857,219.0,7.0,1.0e-55
itemsets s2,0.633,40.277,1.791,150.0,42.0,1.9e-15


method    BS w=10  BS w=20  BS w=5  BS w=50   BUS  BUS +local  itemsets s1  itemsets s1 +guided  itemsets s2  itemsets s2 +guided  pool
k   lits                                                                                                                               
200 3        0.71     0.72    0.64     0.72  0.74        0.74         0.74                 0.74         0.68                 0.72  0.72
    4        0.75     0.75    0.70     0.75  0.86        0.83         0.86                 0.88         0.75                 0.86  0.81
    5        0.67     0.66    0.70     0.66  0.91        0.80         0.91                 0.90         0.69                 0.78  0.75
    6        0.23     0.23    0.29     0.23  0.37        0.24         0.37                 0.32         0.24                 0.21  0.29
800 3        0.48     0.51    0.46     0.52  0.79        0.60         0.79                 0.77         0.68                 0.75  0.61
    4        0.49     0.55    0.48     0.55  0.85        0.68         0.85                 0.79         0.78                 0.82  0.64
    5        0.38     0.37    0.40     0.37  0.85        0.51         0.85                 0.72         0.68                 0.62  0.52
    6        0.16     0.15    0.23     0.15  0.38        0.23         0.38                 0.29         0.30                 0.27  0.18

## 2. Ablation: which part of the fix matters (seed type, description step)

In [3]:
display(table('seeds'))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.524,9225.838,8.316,NaN,NaN,
BUS seed=closest-$,0.000,2282.356,10.090,0.0,227.0,9.3e-69
BUS seed=closest-rel,0.000,2380.275,10.117,0.0,227.0,9.3e-69
BUS seed=closest-blocks,0.000,2613.691,7.197,0.0,227.0,9.3e-69
BUS seed=snn-pairs,0.012,2657.476,5.110,4.0,226.0,1.3e-61
BUS seed=snn-raw,0.707,1071.894,4.004,82.0,3.0,5.3e-21
BUS,0.751,173.219,3.449,100.0,2.0,2.1e-27
BUS describe=share,0.644,2768.806,4.955,54.0,2.0,4.4e-14
BUS describe=all,0.201,4.991,3.203,7.0,147.0,3.3e-35


## 3. Robustness: unknown depth (every method searches to depth 6)

In [4]:
display(table('depth6'))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.522,10217.995,10.775,NaN,NaN,
BS w=20,0.520,19217.275,19.952,6.0,7.0,1.0e+00
BUS,0.751,176.887,3.647,101.0,2.0,1.1e-27
BUS +local,0.619,3432.201,4.100,43.0,1.0,5.1e-12
pool,0.610,786.122,2.349,43.0,5.0,1.4e-08
itemsets s2,0.665,42.813,1.795,82.0,20.0,4.3e-10
itemsets s2 +guided,0.700,125.806,1.839,88.0,11.0,4.5e-16


## 4. Robustness: correlated / hierarchical descriptors

In [5]:
display(table('hier'))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.458,7233.162,7.043,NaN,NaN,
BS w=50,0.477,29054.184,29.185,12.0,3.0,3.5e-02
BUS,0.595,250.376,3.619,73.0,8.0,3.0e-14
BUS +local,0.534,2457.025,3.485,41.0,5.0,4.4e-08
pool,0.504,587.711,2.155,28.0,6.0,2.0e-04
itemsets s2,0.517,110.618,1.796,54.0,26.0,2.3e-03
itemsets s2 +guided,0.532,187.464,1.843,52.0,17.0,2.9e-05


## 5. Robustness: weaker effects and heavy contamination (50% outliers)

In [6]:
display(table("effect"))
display(data["effect"].pivot_table(index=["snr", "outliers"], columns="method", values="found").round(2))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.105,9263.117,8.614,NaN,NaN,
BUS,0.170,120.286,3.793,63.0,20.0,2.4e-06
BUS +local,0.141,5746.305,5.617,38.0,14.0,1.2e-03
pool,0.130,609.220,2.308,31.0,14.0,1.6e-02
itemsets s2 +guided,0.142,49.765,1.887,48.0,23.0,4.1e-03


method        BS w=10   BUS  BUS +local  itemsets s2 +guided  pool
snr outliers                                                      
2   0.0          0.04  0.00        0.02                 0.00  0.01
    0.2          0.00  0.00        0.00                 0.00  0.00
4   0.0          0.15  0.06        0.09                 0.05  0.09
    0.2          0.07  0.03        0.05                 0.02  0.06
8   0.5          0.06  0.23        0.19                 0.21  0.17
16  0.5          0.30  0.70        0.50                 0.57  0.45

## 6. Robustness: a much wider beam (w = 100)

In [7]:
display(table('beam100'))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.514,9084.784,8.459,NaN,NaN,
BS w=100,0.521,65732.421,63.074,6.0,4.0,7.5e-01
BUS,0.774,163.339,3.546,77.0,1.0,5.2e-22


## 7. Sensitivity of BUS's settings (one at a time around k=10, snn=5, u=200, top=8, z=2, maj=0.6, blocks=36)

In [8]:
display(table('sensitivity'))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.524,9225.838,8.175,NaN,NaN,
BUS,0.751,173.219,3.821,100.0,2.0,2.1e-27
BUS k=5,0.000,0.000,2.236,0.0,227.0,9.3e-69
BUS k=20,0.337,49.709,3.552,14.0,95.0,5.8e-16
BUS snn=3,0.513,176.658,4.042,41.0,46.0,6.7e-01
BUS snn=7,0.617,93.589,3.114,78.0,38.0,2.6e-04
BUS u=50,0.746,125.379,3.365,98.0,2.0,8.0e-27
BUS u=400,0.751,173.219,3.507,100.0,2.0,2.1e-27
BUS top=4,0.453,104.111,3.448,43.0,74.0,5.3e-03


## 8. Other targets: Walmart (2,566 × 143 weeks) and Favorita (1,100 × 910 days): the boundary condition

In [9]:
display(table("walmart")); display(table("favorita"))
parts = []
for r in ["main", "walmart", "favorita"]:
    d = data[r][data[r].method.isin(["BUS", "BS w=10"])].copy(); d["run"] = r; parts.append(d)
d = pd.concat(parts)
d["true subgroup size (records)"] = pd.cut(d.planted_n, [0, 30, 60, 100, 200, 5000], labels=["<30", "30-60", "60-100", "100-200", ">200"])
t = d.pivot_table(index=["run", "true subgroup size (records)"], columns="method", values="found", observed=True).round(2)
t["tasks"] = d[d.method == "BUS"].groupby(["run", "true subgroup size (records)"], observed=True).size()
display(t)
diag = []
for r in ["main", "hier", "walmart", "favorita"]:
    b = data[r][data[r].method == "BUS"]
    diag.append({"run": r, "true subgroup (records)": b.planted_n.mean(), "best mini-cluster purity": b.best_mc_purity.mean(),
                 "true conditions in a basket": (b.planted_conds_in_basket / b.lits).mean(),
                 "true description generated": b.evaluated_hit.mean()})
display(pd.DataFrame(diag).set_index("run").round(3))

,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.480,9279.309,4.677,NaN,NaN,
BS w=50,0.491,38979.634,19.750,9.0,4.0,2.7e-01
BUS,0.289,112.043,1.096,14.0,98.0,1.1e-16
BUS +local,0.391,4989.355,2.131,24.0,63.0,3.5e-05
pool,0.305,601.448,0.762,8.0,85.0,2.3e-17
itemsets s2,0.150,21.686,0.537,2.0,147.0,3.1e-41
itemsets s2 +guided,0.189,53.050,0.545,5.0,133.0,2.3e-33


,found,avg evaluations,avg seconds,wins,losses,p (McNemar)
method,,,,,,
BS w=10,0.352,9279.413,3.413,NaN,NaN,
BS w=50,0.356,37773.657,13.912,5.0,3.0,7.3e-01
BUS,0.242,62.811,0.424,18.0,68.0,5.3e-08
BUS +local,0.275,5273.582,1.404,14.0,49.0,1.1e-05
pool,0.235,521.231,0.341,11.0,64.0,3.1e-10
itemsets s2,0.095,9.200,0.206,3.0,120.0,5.8e-32
itemsets s2 +guided,0.127,15.068,0.207,6.0,108.0,2.7e-25


method                                 BS w=10   BUS  tasks
run      true subgroup size (records)                      
favorita <30                              0.23  0.11    376
         30-60                            0.92  0.87     60
         60-100                           1.00  0.93     15
         100-200                          1.00  1.00      4
main     60-100                           0.29  0.61    473
         100-200                          0.68  0.88    277
         >200                             0.96  1.00    125
walmart  <30                              0.20  0.05    122
         30-60                            0.44  0.21    219
         60-100                           0.83  0.67     58
         100-200                          1.00  0.85     33
         >200                             1.00  1.00      8

,true subgroup (records),best mini-cluster purity,true conditions in a basket,true description generated
run,,,,
main,137.413,0.776,0.930,0.817
hier,140.570,0.808,0.930,0.787
walmart,53.011,0.452,0.605,0.316
favorita,22.084,0.438,0.518,0.244
